# 📘 Bài Giảng Thực Hành: Support Vector Regression (SVR)
**Mục tiêu bài giảng:**
1. Hiểu nguyên lý ống $\epsilon$-insensitive và hàm nhân RBF (Radial Basis Function) trong SVR.
2. Nắm vững lý do **BẮT BUỘC phải Chuẩn hóa đặc trưng (Feature Scaling)** cho cả biến $X$ và $y$.
3. Thực hiện chuyển đổi ngược thang đo bằng `inverse_transform` để xuất kết quả tiền tệ thực tế.

---
### 📊 Tập dữ liệu thực hành: `Position_Salaries.csv`


## 1. Nạp Thư viện và Dữ liệu


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVR
from sklearn.metrics import r2_score, root_mean_squared_error

dataset = pd.read_csv('Position_Salaries.csv')
X = dataset.iloc[:, 1:2].values  # Level
y = dataset.iloc[:, 2:3].values  # Salary dạng 2D shape (10, 1) để scale
dataset.head()


📌 **Expected Output:**
DataFrame 10 hàng biểu diễn cấp bậc chức vụ và mức lương.


## 2. Chuẩn Hóa Đặc Trưng (Feature Scaling)


In [ ]:
sc_X = StandardScaler()
sc_y = StandardScaler()

X_scaled = sc_X.fit_transform(X)
y_scaled = sc_y.fit_transform(y)

print('X sau chuẩn hóa (Mean=0, Std=1):\n', np.round(X_scaled.ravel(), 2))
print('y sau chuẩn hóa (Mean=0, Std=1):\n', np.round(y_scaled.ravel(), 2))


📌 **Expected Output:**
Các mảng giá trị Z-score nằm trong khoảng từ -1.5 đến +2.0.


## 3. Huấn luyện Mô hình SVR và Dự đoán Level = 6.5


In [ ]:
regressor = SVR(kernel='rbf', C=1.0, epsilon=0.1)
regressor.fit(X_scaled, y_scaled.ravel())

# Dự đoán Level 6.5
level_scaled = sc_X.transform([[6.5]])
pred_scaled = regressor.predict(level_scaled)
pred_orig = sc_y.inverse_transform(pred_scaled.reshape(-1, 1))[0][0]

# Đánh giá mô hình
y_pred_orig = sc_y.inverse_transform(regressor.predict(X_scaled).reshape(-1, 1))
print(f'Mức lương dự đoán Level 6.5: ${pred_orig:,.2f}')
print(f'R² Score trên dữ liệu gốc:  {r2_score(y, y_pred_orig):.4f}')


📌 **Expected Output:**
```
Mức lương dự đoán Level 6.5: $170,370.02
R² Score trên dữ liệu gốc:  0.7516
```


## 4. Trực quan hóa Đường Cong SVR


In [ ]:
X_grid = np.arange(min(X)[0], max(X)[0], 0.01).reshape(-1, 1)
X_grid_scaled = sc_X.transform(X_grid)
y_grid_pred = sc_y.inverse_transform(regressor.predict(X_grid_scaled).reshape(-1, 1))

plt.figure(figsize=(10, 6))
plt.scatter(X, y, color='red', s=80, label='Dữ liệu Thực tế')
plt.plot(X_grid, y_grid_pred, color='blue', linewidth=2.5, label='Đường cong SVR (RBF Kernel)')
plt.scatter([6.5], [pred_orig], color='purple', s=160, marker='*', zorder=6, label=f'Dự đoán 6.5 (${pred_orig:,.0f})')
plt.title('Position Level vs Salary (Support Vector Regression - SVR)', fontweight='bold')
plt.xlabel('Position Level')
plt.ylabel('Salary ($)')
plt.legend()
plt.show()


📌 **Expected Output:**
Biểu đồ đường cong SVR RBF bám theo xu hướng dữ liệu phi tuyến.
